# Communication Function Annotation Notebook (Gemma API, 5 Dimensions)

This notebook uses an API-style annotation loop. It annotates ads with Gemma only, saves checkpoints, keeps raw outputs and records parse/runtime failures without using fallback labels.

In [1]:
#%pip install -U pandas requests

### Imports 

In [2]:
import json
import re
import sys
import time
from pathlib import Path

import pandas as pd
import requests

### Data loading if you are running on Colab

### I am not using *colab* so mine are comment (please use command + / if you are using mac to uncomment them)

In [3]:
# #this try and except is only useful if you are running the code in both colab or locally, you do not have to comment it neighther way and please add the csv to drive if you use colab
# try:
#     from google.colab import drive  # type: ignore
#     drive.mount('/content/drive', force_remount=False)
# except Exception:
#     pass

# #this fun help you proceed further if you have colab or you run the code locally
# def find_project_root() -> Path:
#   #this checks if you are in colab
#     running_in_colab = 'google.colab' in sys.modules
#     candidates = []
#     if running_in_colab:
#         candidates.append(Path('/content/drive/MyDrive/Gliner-Work.Dauphine'))
#     candidates.append(Path('/Users/raresolteanu/Desktop/Gliner-Work.Dauphine'))

#     for candidate in candidates:
#         if (candidate / 'annotation_working_master_human_2100_seed.csv').exists():
#             return candidate
# #this checks if it is not in the colab it runs it locally
#     here = Path.cwd()
#     for candidate in [here, *here.parents]:
#         if (candidate / 'annotation_working_master_human_2100_seed.csv').exists():
#             return candidate
# #if none were found we set an error to rise
#     raise FileNotFoundError('Could not locate annotation_working_master_human_2100_seed.csv')


# PROJECT_ROOT = find_project_root() #call the fun
# DATASET_PATH = PROJECT_ROOT / 'annotation_working_master_human_2100_seed.csv' #the path to your CSV
# OUTPUT_DIR = PROJECT_ROOT / 'communication_function_outputs_5d'#output path
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True) #do not crash if folder already exists

# #to see if everything work
# print('PROJECT_ROOT =', PROJECT_ROOT)
# print('DATASET_PATH exists =', DATASET_PATH.exists())
# print('OUTPUT_DIR =', OUTPUT_DIR)


### This is the data loading code if both the datasets and your notebook are in the same directory and if you are running it *locally*

In [4]:
PROJECT_ROOT = Path.cwd()
DATASET_PATH = PROJECT_ROOT / "annotation_working_master_human_2100_seed.csv"
OUTPUT_DIR = PROJECT_ROOT / "communication_function_outputs_gemma_api_5d"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATASET_PATH exists =", DATASET_PATH.exists())
print("OUTPUT_DIR =", OUTPUT_DIR)


PROJECT_ROOT = /Users/raresolteanu/Desktop/Gliner-Work.Dauphine
DATASET_PATH exists = True
OUTPUT_DIR = /Users/raresolteanu/Desktop/Gliner-Work.Dauphine/communication_function_outputs_gemma_api_5d


In [5]:
#the columns I want to use from the dataset to give as input to my annotater
TEXT_COLUMNS = [
    'Script',
    'Visuel',
    'Titre',
    'MotsClés',
    'Thème',
]

#the dimnsion that I want to use for annotation
DIMENSIONS = [
    "informativeness",
    "expressiveness",
    "conative",
    "phatic",
    "greenness",
    "creativeness",
    "metalingual",
]


#the scale
SCORE_MIN = 0.0
SCORE_MAX = 5.0
#this is in case of the dominant dimension if two have a score nearly equal the dominant dimension will be mixed
DOMINANT_TIE_TOLERANCE = 0.10

#change the modelm but make sure it is installed on ollama also by using the terminal(you can check with ollama list to see what you have install on it)

MODEL_NAME = "mistral"
 #this is the ligthest gemmav3 version


#the API for ollama
API_URL = 'http://localhost:11434/api/chat'

#This sets the maximum time, in seconds, that the notebook will wait for one API response from Ollama. 
#This needs to be incraesed if you are using more expensive models since more expensive models need more time and you will get to many time errors
REQUEST_TIMEOUT = 600

#this saves the progress every 10 rows
CHECKPOINT_EVERY = 10

#This is the threshold used to decide whether a row should be flagged as low-confidence.
CONFIDENCE_THRESHOLD = 0.0

#quick check of what we did above in this code box 
print('MODEL_NAME =', MODEL_NAME)
print('API_URL =', API_URL)
print('CHECKPOINT_EVERY =', CHECKPOINT_EVERY)

MODEL_NAME = mistral
API_URL = http://localhost:11434/api/chat
CHECKPOINT_EVERY = 10


This handles the missing values from NaN -> "", if not NaN it splits and strips it

In [6]:
def normalize_text(value):
    if pd.isna(value):
        return ''
    return ' '.join(str(value).split()).strip()

It trims long text before sending it to the model.

In [7]:
def shorten_text(text, max_chars=3500):
    text = normalize_text(text)
    if len(text) <= max_chars: #if below 500 it returns it unchanged 
        return text
    return text[:max_chars].rstrip() + ' ...' #keeps only the charcters until the maxchar and put ... to see that the text was cut


This loads the dataset and keeps only what we want

In [8]:
def load_rows(limit=None, review_status='needs_label', row_ids=None):
    #this loads the data and make it numeric
    df = pd.read_csv(DATASET_PATH)
    df['row_id'] = pd.to_numeric(df['row_id'], errors='coerce').astype('Int64')
#keeps only rows where we have review status
    if review_status is not None and 'Review_Status' in df.columns:
        df = df[df['Review_Status'] == review_status].copy()
#this can be comment if you do not need to work with specific rows
    if row_ids is not None:
        wanted = {int(x) for x in row_ids}
        df = df[df['row_id'].isin(wanted)].copy()
#to set the limit
    if limit is not None:
        df = df.head(limit).copy()

    return df

This function checks whether the dataset contains all the columns you expect. This helps to make sure the models can see all the data I want above

In [9]:
def require_columns(df, columns):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f'Missing columns in dataset: {missing}')


This function builds the final ad text that you send to the annotator. Plus it is using the shorten fun to reduce the input

In [10]:
def build_ad_text(row):
    parts = []
#this sets the length of the input
    for col, max_chars in [('Script', 280), ('Visuel', 180), ('Titre', 80), ('MotsClés', 80), ('Thème', 80)]:
#this gets the shorten value or gets '' if there is nothing
        value = shorten_text(row.get(col, ''), max_chars=max_chars)
        if value:
#this make the input dict
            parts.append(f'{col}: {value}')
#this joins the lines together
    return '\n'.join(parts)

In [11]:
preview = load_rows(limit=2, review_status='needs_label')
require_columns(preview, ['row_id'] + TEXT_COLUMNS)
preview[['row_id', 'Marque', 'Produit']].head()
preview = load_rows(limit=2, review_status='needs_label')
#check if we have all the columns
require_columns(preview, ['row_id'] + TEXT_COLUMNS)
#just data visualisation 
preview[['row_id', 'Produit','Script','Visuel','Titre','MotsClés','Thème']].head()


,row_id,Produit,Script,Visuel,Titre,MotsClés,Thème
2100,2100,NISSAN PRESTIGE,"Voix homme : "" NISSAN est fier de vous présent...","Sergio Aguero, nouvel ambassadeur de NISSAN, a...",SERGIO AGUERO,"LIGNE BLANCHE , PIEGE , PREVENTION , DEPASSEME...","AVENTURE, FANTASTIQUE, FOOTBALL, SECURITE, SUC..."
2101,2101,VW PRESTIGE,"Voix homme (1) : "" Il n'y a rien de tel que ri...",De la naissance de l'Univers à l'art contempor...,RIEN,"PUNK , RÉFRIGÉRATEUR , ADOLESCENT , TIMIDITÉ ,...","SEDUCTION, ESPACE, INNOVATION, LANGAGE, ANGLAI..."


In [12]:
RUBRIC_TEXT = """You are scoring automotive advertisements across seven communication dimensions.

Definitions:
- informativeness: factual, referential, technical, offer-based, specification-based, price-based, financing-based, or concrete real-world information. This includes the referential function of language: informing the audience objectively about the product, offer, or situation.
- expressiveness: emotive, aspirational, aesthetic, identity-based, prestige-based, desirability-based, or affective expression. This includes the emotive function of language: expressing mood, desire, tone, feeling, or sender attitude.
- conative: receiver-oriented language designed to attract attention, persuade, instruct, invite, command, encourage, or provoke a reaction from the audience. This includes calls to action, direct appeals, and language clearly aimed at influencing the addressee.
- phatic: relationship-maintaining language, contact, welcome, proximity, companionship, togetherness, greeting, or customer bonding. This dimension should remain narrow: it is about social connection and maintaining contact, not just persuasion or emotional tone.
- greenness: ecological framing, environmental responsibility, cleaner mobility, sustainability, low-emission framing, or respect for the planet. This should capture environmental discourse beyond pure EV technicality when that framing is central.
- creativeness: originality of concept, imaginative storytelling, surprising rhetorical structure, inventive visual framing, metaphorical construction, unusual artistic execution, or foregrounding of the message form itself. This dimension includes most of the poetic function of language: the ad’s crafted style, expressive form, slogan effect, rhetorical artistry, or aesthetic construction.
- metalingual: language that refers to language itself, clarifies wording, explains meaning, comments on expression, or uses self-aware wordplay about language. This dimension will usually be low or absent unless the ad explicitly draws attention to wording or language use.

Scoring scale:
- 0 = absent
- 1 = almost absent
- 2 = weak trace
- 3 = clearly present
- 4 = strong
- 5 = central and unmistakable

Rules:
- Use decimal floats between 0.0 and 5.0 when needed.
- Do not round to integers unless the evidence is truly that clean.
- Ads are not zero-sum. More than one dimension may score high.
- Score each dimension independently.
- Use the ad text and visual description as the main evidence.
- Metadata is only supporting context, not primary evidence.
- Do not give high scores by default.
- A score near 5 should be rare and strongly justified.
- Many ads should have at least one or two low scores.
- Do not assume a car ad is automatically highly informative.
- Give high informativeness only when the ad clearly emphasizes factual information, technical systems, pricing, financing, range, battery, specifications, or concrete product claims.
- Give high expressiveness only when the ad clearly emphasizes desire, beauty, prestige, aspiration, identity, mood, or emotional appeal.
- Give high conative only when the ad clearly pushes the audience toward attention, response, desire, action, or persuasion directed at the receiver.
- Give high phatic only when the ad clearly emphasizes social closeness, bonding, welcome, companionship, direct relational contact, or relationship maintenance.
- Give high greenness only when ecological or environmental framing is clearly central, not just weakly implied.
- Give high creativeness only when the ad idea or execution is clearly original, imaginative, surprising, metaphorical, slogan-like, rhetorically crafted, or artistically unusual.
- Innovation or technology is not automatically creativeness.
- Electrification is not automatically greenness unless ecological framing is central.
- Conative is not the same as phatic: conative tries to influence the receiver, while phatic maintains social contact.
- Creativeness is not the same as expressiveness: creativeness concerns the originality and crafted form of the message, while expressiveness concerns emotional tone or desire.
- Metalingual should usually stay at 0 or 1 unless the ad explicitly comments on wording, language, naming, or verbal expression.
- If two or more dimensions are tied at the top within a tiny margin, use dominant_dimension = mixed.
- dominant_dimension_score must equal the highest of the scored dimensions.
- Keep the reason short and concrete.
- Metalingual is usually 0.0 in normal car advertising.
- Only score metalingual above 0.0 if the ad explicitly reflects on language, wording, naming, meaning, or verbal expression itself.
- Conative is not automatic in advertising.
- Do not score conative highly unless the ad explicitly addresses the receiver, invites action, commands attention, or pushes for a response.
- Use decimal floats between 0.0 and 5.0 when needed.
- Do not round to integers unless the evidence is truly that clean.
- Use the score scale flexibly and precisely: scores do not need to be integers or only end in .5.
- Fine-grained values such as 3.2, 3.7, 4.1, or 4.8 are allowed and encouraged when they better reflect the evidence.
- Do not mechanically default to rounded values like 3.0, 4.0, or 4.5 if a more precise score is justified.
- Evaluate each dimension separately, not globally.
- Do not reuse the same score pattern across rows.
- A high score on one dimension does not imply high scores on the others.
- A low score on one dimension does not imply low scores on the others.
- For each dimension, decide independently whether the evidence is absent, weak, present, strong, or central.
- Only then translate that judgment into a numeric score.
- It is normal for one ad to have one high score and several low scores.
- Avoid template-like outputs such as giving several dimensions the same score unless the evidence clearly supports that.

"""



In [13]:
RUBRIC_TEXT_FR = """Vous évaluez des publicités automobiles selon sept dimensions de communication.

Définitions :
- informativeness : information factuelle, référentielle, technique, liée à l'offre, aux spécifications, au prix, au financement ou à des éléments concrets du monde réel. Cela inclut la fonction référentielle du langage : informer objectivement le public sur le produit, l'offre ou la situation.
- expressiveness : expression émotive, aspirationnelle, esthétique, identitaire, liée au prestige, au désir ou à la tonalité affective. Cela inclut la fonction émotive du langage : exprimer une humeur, un désir, un ton, un sentiment ou l'attitude de l'émetteur.
- conative : langage orienté vers le destinataire pour attirer l'attention, persuader, instruire, inviter, inciter, commander, encourager ou provoquer une réaction. Cela inclut les appels à l'action, les interpellations directes et les formulations qui cherchent clairement à influencer le récepteur.
- phatic : langage de maintien du lien, de contact, d'accueil, de proximité, de compagnie, de convivialité, de salutation ou de lien relationnel avec le client. Cette dimension doit rester étroite : elle concerne la connexion sociale et le maintien du contact, pas simplement la persuasion ou la tonalité émotionnelle.
- greenness : cadrage écologique, responsabilité environnementale, mobilité plus propre, durabilité, faibles émissions ou respect de la planète. Cette dimension doit capter un discours environnemental allant au-delà de la seule technicité des véhicules électriques lorsque ce cadrage est central.
- creativeness : originalité du concept, narration imaginative, structure rhétorique surprenante, cadrage visuel inventif, construction métaphorique, exécution artistique inhabituelle ou mise en avant de la forme même du message. Cette dimension recouvre une grande partie de la fonction poétique du langage : style travaillé, forme expressive, effet de slogan, art rhétorique ou construction esthétique.
- metalingual : langage qui renvoie au langage lui-même, clarifie une formulation, explique un sens, commente l'expression ou utilise un jeu de mots autoréflexif. Cette dimension sera généralement faible ou absente sauf si la publicité attire explicitement l'attention sur les mots, le langage ou leur usage.

Échelle de score :
- 0 = absent
- 1 = presque absent
- 2 = faible trace
- 3 = clairement présent
- 4 = fort
- 5 = central et incontestable

Règles :
- Utilisez des nombres décimaux entre 0.0 et 5.0 lorsque c'est utile.
- N'arrondissez pas à des entiers sauf si l'évidence est réellement aussi nette.
- Les publicités ne sont pas à somme nulle. Plusieurs dimensions peuvent avoir un score élevé.
- Évaluez chaque dimension indépendamment.
- Utilisez principalement le texte de la publicité et la description visuelle comme base d'évaluation.
- Les métadonnées ne sont qu'un contexte d'appui, pas une preuve principale.
- N'attribuez pas de scores élevés par défaut.
- Un score proche de 5 doit être rare et fortement justifié.
- Beaucoup de publicités devraient avoir au moins une ou deux dimensions faibles.
- Ne supposez pas qu'une publicité automobile est automatiquement très informative.
- N'attribuez un score élevé à informativeness que si la publicité met clairement en avant des informations factuelles, des systèmes techniques, des prix, du financement, l'autonomie, la batterie, des spécifications ou des promesses produit concrètes.
- N'attribuez un score élevé à expressiveness que si la publicité met clairement en avant le désir, la beauté, le prestige, l'aspiration, l'identité, l'humeur ou l'appel émotionnel.
- N'attribuez un score élevé à conative que si la publicité pousse clairement le public vers l'attention, la réaction, le désir, l'action ou une persuasion orientée vers le destinataire.
- N'attribuez un score élevé à phatic que si la publicité met clairement en avant la proximité sociale, le lien, l'accueil, la compagnie, le contact relationnel direct ou le maintien de la relation.
- N'attribuez un score élevé à greenness que si le cadrage écologique ou environnemental est clairement central, et non simplement suggéré faiblement.
- N'attribuez un score élevé à creativeness que si l'idée ou l'exécution de la publicité est clairement originale, imaginative, surprenante, métaphorique, travaillée comme un slogan, rhétoriquement construite ou artistiquement inhabituelle.
- L'innovation ou la technologie ne sont pas automatiquement de la creativeness.
- L'électrification n'est pas automatiquement de la greenness sauf si le cadrage écologique est central.
- Conative n'est pas la même chose que phatic : conative cherche à influencer le destinataire, tandis que phatic maintient le lien social.
- Creativeness n'est pas la même chose que expressiveness : creativeness concerne l'originalité et la forme travaillée du message, tandis que expressiveness concerne la tonalité émotionnelle ou le désir.
- Metalingual doit en général rester à 0 ou 1 sauf si la publicité commente explicitement la formulation, le langage, le nommage ou l'expression verbale.
- Si deux dimensions ou plus sont à égalité au sommet dans une marge très faible, utilisez dominant_dimension = mixed.
- dominant_dimension_score doit être égal au score le plus élevé parmi les dimensions.
- Gardez la raison courte et concrète.
- Metalingual vaut généralement 0.0 dans la publicité automobile ordinaire.
- N'attribuez un score supérieur à 0.0 à metalingual que si la publicité réfléchit explicitement au langage, aux mots, au nommage, au sens ou à l'expression verbale elle-même.
- Conative n'est pas automatique dans la publicité.
- N'attribuez pas un score élevé à conative sauf si la publicité s'adresse explicitement au destinataire, invite à l'action, commande l'attention ou pousse à une réponse.
- Utilisez l'échelle de score de façon souple et précise : les scores n'ont pas besoin d'être des entiers ni de se terminer uniquement par .5.
- Des valeurs fines comme 3.2, 3.7, 4.1 ou 4.8 sont autorisées et encouragées lorsqu'elles reflètent mieux l'évidence.
- Ne tombez pas mécaniquement dans des valeurs arrondies comme 3.0, 4.0 ou 4.5 si un score plus précis est justifié.
- Évaluez chaque dimension séparément, et non de manière globale.
- Ne réutilisez pas le même schéma de scores d'une ligne à l'autre.
- Un score élevé sur une dimension n'implique pas des scores élevés sur les autres.
- Un score faible sur une dimension n'implique pas des scores faibles sur les autres.
- Pour chaque dimension, décidez indépendamment si l'évidence est absente, faible, présente, forte ou centrale.
- Traduisez seulement ensuite ce jugement en score numérique.
- Il est normal qu'une publicité ait une dimension élevée et plusieurs dimensions faibles.
- Évitez les sorties stéréotypées, par exemple attribuer les mêmes scores à plusieurs dimensions, sauf si l'évidence le justifie clairement.

"""


In [14]:
SUPERVISOR_NOTES = """Supervisor notes:
- informative content has a social use because it informs people;
- persuasive and desirability-oriented content is closer to expressiveness;
- phatic language maintains an affective relation with the customer;
- greenness should capture ecological framing beyond pure EV technicality when that framing is central;
- creativeness should capture originality of the ad idea, not just whether the product itself is innovative.
"""

In [15]:
EXAMPLES_TEXT_ENG = """Examples by dimension.

Single-dimension examples

informativeness
- "Up to 620 km of range on one charge."
- "Lease from 299 euros per month."
- "Fast charging from 10% to 80% in 28 minutes."
- "Five-year warranty included."
- "Available in hybrid, plug-in hybrid, and electric versions."

expressiveness
- "A car made to move your heart."
- "Pure elegance in every line."
- "Feel the thrill of every journey."
- "Designed for those who dare to desire more."
- "An irresistible presence on every street."

conative
- "Book your test drive today."
- "Discover the new model now."
- "Choose the future of driving."
- "Step inside and feel the difference."
- "Visit your nearest dealership this weekend."

phatic
- "We are always by your side."
- "Welcome to the family."
- "Together on every road."
- "Here for you, every day."
- "See you soon in our showroom."

greenness
- "Drive toward a cleaner tomorrow."
- "Lower emissions for everyday mobility."
- "Built with sustainability in mind."
- "A responsible choice for the road ahead."
- "Cleaner mobility for modern cities."

creativeness
- "Silence with a pulse."
- "The city bends around your motion."
- "Not just a car, but a dream in motion."
- "Where steel learns to breathe."
- "A road trip written like a love letter."

metalingual
- "By 'freedom' we mean confidence to go farther."
- "When we say 'clean', we mean cleaner for city life."
- "Redefining what 'performance' means."
- "Call it a car if you want, we call it a language of motion."
- "This is not just electric, it is our definition of progress."

Two-dimension combination examples

informativeness + expressiveness
- "Advanced hybrid technology, wrapped in breathtaking design."
- "Technical precision with a presence you will never forget."

informativeness + conative
- "Discover up to 620 km of range today."
- "Book your test drive and experience our fastest charging ever."

informativeness + phatic
- "We are here to guide you through every feature."
- "Our team is with you at every step of your electric journey."

informativeness + greenness
- "Lower emissions and up to 620 km of range."
- "Cleaner mobility with efficient hybrid technology."

informativeness + creativeness
- "Engineering that turns every kilometer into a story."
- "Precision technology, imagined differently."

informativeness + metalingual
- "When we say 'range', we mean freedom without compromise."
- "By 'efficiency' we mean more distance with less waste."

expressiveness + conative
- "Feel the difference, book your drive today."
- "Choose the car that speaks to your ambition."

expressiveness + phatic
- "Welcome to a driving experience made for you."
- "Together, let desire meet everyday life."

expressiveness + greenness
- "Beautiful design for a more responsible future."
- "Drive what you love, with greater respect for the planet."

expressiveness + creativeness
- "A dream sculpted in silence."
- "Beauty written as motion."

expressiveness + metalingual
- "This is what we mean by pure desire."
- "Redefining elegance, one line at a time."

conative + phatic
- "Come see us, we are ready to welcome you."
- "Join us this weekend and let us guide you."

conative + greenness
- "Choose cleaner driving today."
- "Make the responsible move now."

conative + creativeness
- "Step into a new idea of motion."
- "Discover the car that rewrites the road."

conative + metalingual
- "Rethink what 'driving pleasure' really means."
- "See how we redefine performance today."

phatic + greenness
- "Together for a cleaner tomorrow."
- "With you on the road to more responsible mobility."

phatic + creativeness
- "Welcome to a new story of driving."
- "Together, let the road become poetry."

phatic + metalingual
- "When we say 'together', we mean more than sharing a ride."
- "Here for you, in every sense of the word."

greenness + creativeness
- "A cleaner future imagined beautifully."
- "Sustainability with a poetic heartbeat."

greenness + metalingual
- "By 'cleaner', we mean better for the city and the air."
- "This is how we redefine responsible mobility."

creativeness + metalingual
- "Not just a slogan, but a new language of motion."
- "When style starts speaking for itself."
"""


In [16]:
EXAMPLES_TEXT_FR = """Exemples par dimension.

Exemples à dominante unique

informativeness
- "Jusqu'à 620 km d'autonomie avec une seule charge."
- "Location à partir de 299 euros par mois."
- "Recharge rapide de 10 % à 80 % en 28 minutes."
- "Garantie de cinq ans incluse."
- "Disponible en version hybride, hybride rechargeable et électrique."

expressiveness
- "Une voiture conçue pour faire battre votre coeur."
- "Une élégance pure dans chaque ligne."
- "Ressentez l'intensité de chaque trajet."
- "Pensée pour ceux qui osent désirer davantage."
- "Une présence irrésistible à chaque coin de rue."

conative
- "Réservez votre essai dès aujourd'hui."
- "Découvrez le nouveau modèle maintenant."
- "Choisissez l'avenir de la conduite."
- "Montez à bord et sentez la différence."
- "Rendez-vous chez votre concessionnaire ce week-end."

phatic
- "Nous sommes toujours à vos côtés."
- "Bienvenue dans la famille."
- "Ensemble sur toutes les routes."
- "Là pour vous, chaque jour."
- "À très bientôt dans notre showroom."

greenness
- "Roulez vers un avenir plus propre."
- "Des émissions réduites pour la mobilité quotidienne."
- "Conçue dans un esprit de durabilité."
- "Un choix responsable pour la route de demain."
- "Une mobilité plus propre pour les villes modernes."

creativeness
- "Le silence a désormais un battement."
- "La ville se plie à votre mouvement."
- "Pas seulement une voiture, mais un rêve en mouvement."
- "Là où l'acier apprend à respirer."
- "Un voyage écrit comme une lettre d'amour."

metalingual
- "Par 'liberté', nous entendons la confiance d'aller plus loin."
- "Quand nous disons 'propre', nous parlons d'une ville plus respirable."
- "Redéfinir ce que signifie vraiment la performance."
- "Appelez-la voiture si vous voulez, nous y voyons un langage du mouvement."
- "Ce n'est pas seulement électrique, c'est notre définition du progrès."

Exemples de combinaisons à deux dimensions

informativeness + expressiveness
- "Une technologie hybride avancée, enveloppée dans un design saisissant."
- "Une précision technique avec une présence que vous n'oublierez jamais."

informativeness + conative
- "Découvrez dès aujourd'hui jusqu'à 620 km d'autonomie."
- "Réservez votre essai et découvrez notre recharge la plus rapide."

informativeness + phatic
- "Nous sommes là pour vous guider à travers chaque fonctionnalité."
- "Notre équipe est avec vous à chaque étape de votre parcours électrique."

informativeness + greenness
- "Des émissions réduites et jusqu'à 620 km d'autonomie."
- "Une mobilité plus propre grâce à une technologie hybride efficace."

informativeness + creativeness
- "Une ingénierie qui transforme chaque kilomètre en histoire."
- "La précision technologique, imaginée autrement."

informativeness + metalingual
- "Quand nous parlons d'autonomie, nous parlons de liberté sans compromis."
- "Par 'efficacité', nous entendons plus de distance avec moins de gaspillage."

expressiveness + conative
- "Ressentez la différence, réservez votre essai dès aujourd'hui."
- "Choisissez la voiture qui parle à votre ambition."

expressiveness + phatic
- "Bienvenue dans une expérience de conduite pensée pour vous."
- "Ensemble, faisons se rencontrer le désir et le quotidien."

expressiveness + greenness
- "Un design magnifique pour un avenir plus responsable."
- "Aimez ce que vous conduisez, avec plus de respect pour la planète."

expressiveness + creativeness
- "Un rêve sculpté dans le silence."
- "La beauté écrite comme un mouvement."

expressiveness + metalingual
- "Voilà ce que nous entendons par pur désir."
- "Redéfinir l'élégance, ligne après ligne."

conative + phatic
- "Venez nous voir, nous sommes prêts à vous accueillir."
- "Rejoignez-nous ce week-end et laissez-nous vous guider."

conative + greenness
- "Choisissez dès aujourd'hui une conduite plus propre."
- "Faites maintenant le choix responsable."

conative + creativeness
- "Entrez dans une nouvelle idée du mouvement."
- "Découvrez la voiture qui réécrit la route."

conative + metalingual
- "Repensez ce que signifie vraiment le plaisir de conduire."
- "Découvrez comment nous redéfinissons la performance aujourd'hui."

phatic + greenness
- "Ensemble pour un avenir plus propre."
- "À vos côtés sur la route d'une mobilité plus responsable."

phatic + creativeness
- "Bienvenue dans une nouvelle histoire de la conduite."
- "Ensemble, faisons de la route une poésie."

phatic + metalingual
- "Quand nous disons 'ensemble', cela veut dire plus que partager un trajet."
- "Là pour vous, dans tous les sens du terme."

greenness + creativeness
- "Un avenir plus propre, imaginé avec beauté."
- "La durabilité avec un souffle poétique."

greenness + metalingual
- "Par 'plus propre', nous entendons meilleur pour la ville et pour l'air."
- "Voilà comment nous redéfinissons la mobilité responsable."

creativeness + metalingual
- "Pas seulement un slogan, mais un nouveau langage du mouvement."
- "Quand le style commence à parler de lui-même."
"""


In [17]:
EXAMPLES_PER_DIM_ENG = """Score calibration by dimension.

These examples are not labels for your dataset. They are calibration cues showing how the wording might sound at different score levels.

informativeness
- 0.0: "A new feeling on every road."
- 1.0: "A new model for modern drivers."
- 2.0: "Available in hybrid and electric versions."
- 3.0: "Offers hybrid technology and lower fuel consumption."
- 3.5: "Hybrid power, lower emissions, and practical everyday efficiency."
- 4.0: "Up to 520 km of range and rapid charging in 30 minutes."
- 4.5: "Available from 299 euros per month with 520 km of range and fast charging."
- 5.0: "625 km range, 10 to 80% charging in 28 minutes, five-year warranty, and lease from 299 euros."

expressiveness
- 0.0: "Now available in dealerships."
- 1.0: "A stylish choice for everyday driving."
- 2.0: "Elegant lines and a refined presence."
- 3.0: "A design that makes every journey feel special."
- 3.5: "A presence that turns everyday driving into desire."
- 4.0: "Pure elegance built to stir emotion."
- 4.5: "A car that speaks to desire, beauty, and identity."
- 5.0: "An irresistible expression of desire, elegance, and emotional power."

conative
- 0.0: "The model is available now."
- 1.0: "Ready for those who want more."
- 2.0: "See the new model in showroom."
- 3.0: "Discover the new model today."
- 3.5: "Visit your dealership and experience the difference."
- 4.0: "Book your test drive now and choose electric confidence."
- 4.5: "Take the wheel today and make the switch."
- 5.0: "Act now, book your test drive, and choose the future of driving."

phatic
- 0.0: "Fast charging in 28 minutes."
- 1.0: "Made for everyday life."
- 2.0: "Always close to you."
- 3.0: "We are with you on every road."
- 3.5: "Together, we make every journey easier."
- 4.0: "Welcome to the family, we are always by your side."
- 4.5: "Here for you, with you, every step of the way."
- 5.0: "Welcome, stay with us, and feel supported on every journey."

greenness
- 0.0: "A bold new driving experience."
- 1.0: "Modern mobility for modern life."
- 2.0: "A lower-emission option for city driving."
- 3.0: "Cleaner mobility designed for everyday use."
- 3.5: "A more responsible drive with lower emissions and cleaner intent."
- 4.0: "Designed for cleaner mobility and a more sustainable future."
- 4.5: "A strong commitment to low-emission, responsible driving."
- 5.0: "A central promise of cleaner mobility, sustainability, and environmental responsibility."

creativeness
- 0.0: "The new SUV is here."
- 1.0: "A fresh look for the road."
- 2.0: "A slightly distinctive visual identity."
- 3.0: "An ad with a noticeable conceptual twist."
- 3.5: "An imaginative framing that gives the message a memorable shape."
- 4.0: "A clearly original execution with unusual visual or rhetorical construction."
- 4.5: "A strikingly inventive concept that reshapes how the product is presented."
- 5.0: "A highly original, metaphorical, and artistically crafted execution that foregrounds the form of the message."

metalingual
- 0.0: "Drive farther with confidence."
- 1.0: "A light suggestion that this is a new language of motion."
- 2.0: "A mild play on the meaning of words like freedom or performance."
- 3.0: "The ad partly explains or redefines a key word it uses."
- 3.5: "The message actively clarifies what a word or expression means in context."
- 4.0: "The ad strongly comments on naming, wording, or expression itself."
- 4.5: "The ad is highly self-aware about the language it uses and its meaning."
- 5.0: "The ad centrally revolves around explaining, redefining, or playing with language itself."

Important calibration rules
- Scores of 4.0 and above should be uncommon and strongly justified.
- Scores of 0.0, 1.0, and 2.0 are normal and should be used often when evidence is weak.
- Do not assign 3.0 or 4.0 by default.
- Do not assign metalingual above 0.0 unless the ad explicitly comments on language, meaning, naming, or wording.
- Do not assign mixed unless two top dimensions are both strong and genuinely almost equal.
"""


In [18]:
EXAMPLES_PER_DIM_FR = """Calibration des scores par dimension.

Ces exemples ne sont pas des labels de votre dataset. Ils servent seulement à montrer à quoi peut ressembler le langage correspondant à différents niveaux de score.

informativeness
- 0.0: "Une nouvelle sensation sur toutes les routes."
- 1.0: "Un nouveau modèle pour les conducteurs d'aujourd'hui."
- 2.0: "Disponible en versions hybride et électrique."
- 3.0: "Propose une technologie hybride et une consommation réduite."
- 3.5: "Une motorisation hybride, moins d'émissions et une efficacité adaptée au quotidien."
- 4.0: "Jusqu'à 520 km d'autonomie et recharge rapide en 30 minutes."
- 4.5: "Disponible à partir de 299 euros par mois avec 520 km d'autonomie et recharge rapide."
- 5.0: "625 km d'autonomie, recharge de 10 % à 80 % en 28 minutes, garantie de cinq ans et location à partir de 299 euros."

expressiveness
- 0.0: "Disponible dès maintenant en concession."
- 1.0: "Un choix élégant pour la conduite au quotidien."
- 2.0: "Des lignes élégantes et une présence raffinée."
- 3.0: "Un design qui rend chaque trajet plus spécial."
- 3.5: "Une présence qui transforme le quotidien en désir."
- 4.0: "Une élégance pure conçue pour susciter l'émotion."
- 4.5: "Une voiture qui parle au désir, à la beauté et à l'identité."
- 5.0: "Une expression irrésistible du désir, de l'élégance et de la puissance émotionnelle."

conative
- 0.0: "Le modèle est désormais disponible."
- 1.0: "Prêt pour ceux qui veulent davantage."
- 2.0: "Découvrez le nouveau modèle en concession."
- 3.0: "Découvrez le nouveau modèle dès aujourd'hui."
- 3.5: "Rendez-vous chez votre concessionnaire et vivez la différence."
- 4.0: "Réservez votre essai maintenant et choisissez la confiance électrique."
- 4.5: "Prenez le volant aujourd'hui et passez à l'étape suivante."
- 5.0: "Agissez maintenant, réservez votre essai et choisissez l'avenir de la conduite."

phatic
- 0.0: "Recharge rapide en 28 minutes."
- 1.0: "Pensé pour la vie quotidienne."
- 2.0: "Toujours proche de vous."
- 3.0: "Nous sommes avec vous sur chaque route."
- 3.5: "Ensemble, nous rendons chaque trajet plus simple."
- 4.0: "Bienvenue dans la famille, nous sommes toujours à vos côtés."
- 4.5: "Là pour vous, avec vous, à chaque étape."
- 5.0: "Bienvenue, restez avec nous et sentez-vous accompagné à chaque trajet."

greenness
- 0.0: "Une nouvelle expérience de conduite audacieuse."
- 1.0: "Une mobilité moderne pour la vie moderne."
- 2.0: "Une option à faibles émissions pour la ville."
- 3.0: "Une mobilité plus propre pensée pour le quotidien."
- 3.5: "Une conduite plus responsable avec moins d'émissions et une intention plus propre."
- 4.0: "Conçue pour une mobilité plus propre et un avenir plus durable."
- 4.5: "Un engagement fort envers une conduite responsable à faibles émissions."
- 5.0: "Une promesse centrale de mobilité plus propre, de durabilité et de responsabilité environnementale."

creativeness
- 0.0: "Le nouveau SUV est arrivé."
- 1.0: "Un regard nouveau sur la route."
- 2.0: "Une identité visuelle légèrement distinctive."
- 3.0: "Une publicité avec une touche conceptuelle perceptible."
- 3.5: "Une mise en scène imaginative qui donne au message une forme mémorable."
- 4.0: "Une exécution clairement originale avec une construction visuelle ou rhétorique inhabituelle."
- 4.5: "Un concept fortement inventif qui transforme la manière de présenter le produit."
- 5.0: "Une exécution très originale, métaphorique et artistiquement construite qui met au premier plan la forme du message."

metalingual
- 0.0: "Allez plus loin en toute confiance."
- 1.0: "Une légère suggestion d'un nouveau langage du mouvement."
- 2.0: "Un léger jeu sur le sens de mots comme liberté ou performance."
- 3.0: "La publicité explique en partie ou redéfinit un mot-clé qu'elle utilise."
- 3.5: "Le message clarifie activement ce qu'un mot ou une expression signifie dans ce contexte."
- 4.0: "La publicité commente fortement le nom, le choix des mots ou l'expression elle-même."
- 4.5: "La publicité est très consciente du langage qu'elle utilise et de son sens."
- 5.0: "La publicité repose de façon centrale sur l'explication, la redéfinition ou le jeu avec le langage lui-même."

Règles importantes de calibration
- Les scores de 4.0 et plus doivent être rares et fortement justifiés.
- Les scores de 0.0, 1.0 et 2.0 sont normaux et doivent être utilisés souvent lorsque l'évidence est faible.
- N'attribuez pas 3.0 ou 4.0 par défaut.
- N'attribuez pas metalingual au-dessus de 0.0 sauf si la publicité commente explicitement le langage, le sens, le nommage ou le choix des mots.
- N'attribuez pas mixed sauf si les deux dimensions supérieures sont toutes les deux fortes et réellement presque égales.
"""


how I want my output (JSON) to look like from the annotater

In [19]:
SCHEMA = {
    "informativeness": "float from 0.0 to 5.0",
    "expressiveness": "float from 0.0 to 5.0",
    "conative": "float from 0.0 to 5.0",
    "phatic": "float from 0.0 to 5.0",
    "greenness": "float from 0.0 to 5.0",
    "creativeness": "float from 0.0 to 5.0",
    "metalingual": "float from 0.0 to 5.0",
    "dominant_dimension": "exactly one of: informativeness, expressiveness, conative, phatic, greenness, creativeness, metalingual, mixed",
    "dominant_dimension_score": "float from 0.0 to 5.0 and equal to the highest dimension score",
    "reason": "short explanation, ideally under 12 words",
    "confidence": "float from 0.0 to 1.0"
}


The actual prompt

In [20]:
SYSTEM_PROMPT = f"""You are an expert annotator of automotive advertisements.
You score five communication dimensions.

{RUBRIC_TEXT}

Vous êtes un annotateur expert de publicités automobiles.
Évaluez chaque publicité selon les dimensions de communication et retournez uniquement un objet JSON valide.
{RUBRIC_TEXT_FR}

{SUPERVISOR_NOTES}

These are calibration examples in English.
They are not labels from the dataset.
Use them only to understand how each communication dimension sounds in language and how dimensions can combine.

{EXAMPLES_TEXT_ENG}

Voici des exemples de calibration en français.
Ce ne sont pas des labels du dataset.
Utilisez-les uniquement pour comprendre à quoi ressemble chaque dimension communicationnelle dans le langage et comment certaines dimensions peuvent se combiner.

{EXAMPLES_TEXT_FR}

These are score calibration examples.
They show what different score levels may look like in language for each dimension.

{EXAMPLES_PER_DIM_ENG}

Voici des exemples de calibration des scores.
Ils montrent à quoi peuvent ressembler différents niveaux de score dans le langage pour chaque dimension.

{EXAMPLES_PER_DIM_FR}



Return only one valid JSON object.
Do not write markdown.
Do not use code fences.
Do not write any text before or after the JSON.

Schema:
{json.dumps(SCHEMA, ensure_ascii=False, indent=2)}

Rules:
- All five scores must be floats from 0.0 to 5.0
- dominant_dimension must be exactly one of: informativeness, expressiveness, phatic, greenness, creativeness, mixed
- dominant_dimension_score must equal the highest score
- confidence must be a float from 0.0 to 1.0
- reason must be short
"""

this function builds the user message sent to the model by placing the ad text inside a simple annotation prompt.

In [21]:
def build_user_prompt(row):
    return f"""Annotate this advertisement.

Ad:
{build_ad_text(row)}
"""

In [22]:
def extract_json(text):
    text = str(text).strip()
#this is looking for JSON object inside a fenced code block
    fenced = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if fenced:
        try:
            return json.loads(fenced.group(1))
        except Exception:
            pass

    match = re.search(r'\{.*\}', text, re.DOTALL)


#this tries to parse that whole block as JSON.

    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass

    return None

In case the models gives different names for the columns 

In [23]:
def normalize_dimension_name(value):
    text = str(value or "").strip().lower()
    aliases = {
        "referential": "informativeness",
        "informative": "informativeness",
        "information": "informativeness",

        "emotive": "expressiveness",
        "expressive": "expressiveness",
        "emotion": "expressiveness",

        "conative": "conative",
        "directive": "conative",
        "persuasive": "conative",

        "phatic": "phatic",
        "phatique": "phatic",

        "green": "greenness",
        "greeness": "greenness",
        "ecology": "greenness",
        "ecological": "greenness",
        "sustainability": "greenness",

        "creative": "creativeness",
        "creativity": "creativeness",
        "poetic": "creativeness",
        "aesthetic": "creativeness",
        "originality": "creativeness",

        "metalingual": "metalingual",
        "metalinguistic": "metalingual",
        "meta": "metalingual",
    }
    text = aliases.get(text, text)
    return text if text in DIMENSIONS or text == "mixed" else ""


This function makes sure a score is a valid number between your minimum and maximum.

In [24]:
def clamp_score(value):
    try:
        score = float(value) #this converts it in floot
    except Exception:
        score = SCORE_MIN #####should ask the supervisor if that is good or it should be change to NaN
#this forces the results between min and max
    return max(SCORE_MIN, min(SCORE_MAX, score)) 

This function decides which dimension is dominant.

In [25]:
def pick_dominant_dimension(scores):
  #this sorts it
    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    top_label, top_score = ranked[0]

  #the rest of code checks the mixed cases
    same_top = [label for label, score in ranked if abs(score - top_score) <= DOMINANT_TIE_TOLERANCE]
    if len(same_top) > 1:
        return 'mixed', top_score
    return top_label, top_score

This function takes the parsed JSON from the model and cleans it.

In [26]:
def validate_payload(parsed):
    out = {}
#this loops through all dim and for each one gets the model output and it makes it between 0.0 and 5.0
    for dim in DIMENSIONS:
        out[dim] = clamp_score(parsed.get(dim))

#this picks a dominant dim
    inferred_dim, inferred_score = pick_dominant_dimension({dim: out[dim] for dim in DIMENSIONS})
#this take into account diff names 
#we keep the raw model label if you want to inspect it later, but we do not trust it for the final output
    model_dominant_dimension = normalize_dimension_name(parsed.get('dominant_dimension'))
#sets all scores between 0.0 and 5.0
    model_dominant_dimension_score = clamp_score(parsed.get('dominant_dimension_score'))
#checks whether the model’s dominant score matches the highest actual dimension score. 
#for the final output we force consistency with the real top score from the five dimensions
    dominant_dimension = inferred_dim
    dominant_dimension_score = inferred_score

#outputs
    out['dominant_dimension'] = dominant_dimension
    out['dominant_dimension_score'] = dominant_dimension_score
    out['reason'] = ' '.join(str(parsed.get('reason', '')).split()).strip()[:200]

#confidence this can be deleted or commented if you are not using confidence
    try:
        out['confidence'] = max(0.0, min(1.0, float(parsed.get('confidence', 0.0))))
    except Exception:
        out['confidence'] = 0.0

#optional raw model dominant fields for debugging
    out['model_dominant_dimension_raw'] = model_dominant_dimension
    out['model_dominant_dimension_score_raw'] = model_dominant_dimension_score

    return out


A function that annotates one dataset row using Gemma through the API.

In [27]:
def annotate_row_with_gemma(row):
#builds the chat messages sent to the model.The prompt
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_prompt(row)},
    ]

#Sends an HTTP POST request to the Ollama API endpoint.
    resp = requests.post(
        API_URL,
        json={
            'model': MODEL_NAME,
            'messages': messages,
            'stream': False, #asks for one full response, not streamed chunks
            'options': {'temperature': 0.0}, #deterministic output
        },
        timeout=REQUEST_TIMEOUT, #Sets how long Python waits before giving up on the request.
    )
    resp.raise_for_status()

#reads the API response as JSON and extracts the model’s generated text.
    generated = resp.json()['message']['content'] #this is the raw answer from Gemma
#this tries to extract a JSON object from the generated text.
    parsed = extract_json(generated)
    if parsed is None:
#for errors, failed rows are kept for inspection.
        return {
            'row_id': int(row['row_id']),
            'parse_ok': False,
            'error': 'Could not parse JSON',
            'raw_output': generated,
        }

#output
    validated = validate_payload(parsed)
    validated['row_id'] = int(row['row_id'])
    validated['model_name'] = MODEL_NAME
    validated['parse_ok'] = True
    validated['raw_output'] = generated
    validated['flagged'] = validated['confidence'] < CONFIDENCE_THRESHOLD
    return validated

### Quick sanity check

In [28]:
#loads one row from the dataset.
example = load_rows(limit=1, review_status='needs_label').iloc[0]
example_result = annotate_row_with_gemma(example)
#this keeps everything as dict besides the raw answer which is not useful for annotation
{k: v for k, v in example_result.items() if k != 'raw_output'}

{'informativeness': 1.0,
 'expressiveness': 2.0,
 'conative': 0.0,
 'phatic': 0.0,
 'greenness': 0.0,
 'creativeness': 3.5,
 'metalingual': 0.0,
 'dominant_dimension': 'creativeness',
 'dominant_dimension_score': 3.5,
 'reason': 'Ad features a new ambassador and a creative football-themed scenario',
 'confidence': 0.9,
 'model_dominant_dimension_raw': 'creativeness',
 'model_dominant_dimension_score_raw': 3.5,
 'row_id': 2100,
 'model_name': 'mistral',
 'parse_ok': True,
 'flagged': False}

The main pipeline function.

In [30]:
def run_annotation_pipeline(limit=10, review_status='needs_label'):
#loads the rows you want to annotate.
    source_rows = load_rows(limit=limit, review_status=review_status).copy().reset_index(drop=True)
    total = len(source_rows)

#defines the output file paths.
    checkpoint_path = OUTPUT_DIR / 'gemma_api_checkpoint.csv'
    final_path = OUTPUT_DIR / 'gemma_api_annotations.csv'
    error_path = OUTPUT_DIR / 'gemma_api_errors.csv'

    results = []
    errors = []

#stores the start time of the whole run.
#this is used later to compute average time and ETA.
    run_start = time.time()

    for i, row in source_rows.iterrows():
        t0 = time.time() #stores the start time
#tries to annotate the row with Gemma.      
        try:
            result = annotate_row_with_gemma(row)
            if result.get('parse_ok'): #only if the row was parsed successfully:
                results.append(result)
            else: #if not add it to errors
                errors.append(result)
        except Exception as exc:
#adds a structured error record for that row.
            errors.append({
                'row_id': int(row['row_id']),
                'parse_ok': False,
                'error': str(exc),
                'raw_output': '',
            })
#computes how long this row took.
        elapsed = time.time() - t0
        done = i + 1 #counts how many rows have been processed so far.
        avg = (time.time() - run_start) / done #avg of them
        eta = avg * (total - done) / 60 if done < total else 0 #estimates the remaining time in minutes.
#prints live progress information, for example:    
        print(f'{done}/{total} {elapsed:.1f}s avg {avg:.1f}s ETA {eta:.1f} min')

#every CHECKPOINT_EVERY rows, save progress.
        if done % CHECKPOINT_EVERY == 0:
            pd.DataFrame(results).to_csv(checkpoint_path, index=False)
            pd.DataFrame(errors).to_csv(error_path, index=False)
            print(f'checkpoint saved at {done}/{total}')

#converts them into DF
    results_df = pd.DataFrame(results)
    errors_df = pd.DataFrame(errors)

    results_df.to_csv(final_path, index=False)
    errors_df.to_csv(error_path, index=False)

#prints the output file locations.
    print('wrote', final_path)
    print('wrote', error_path)
    return results_df, errors_df

#Then this line actually runs the pipeline:
results_df, errors_df = run_annotation_pipeline(limit=10, review_status='needs_label')
results_df.head()

1/10 41.8s avg 41.8s ETA 6.3 min
2/10 41.0s avg 41.5s ETA 5.5 min
3/10 43.0s avg 42.0s ETA 4.9 min
4/10 41.3s avg 41.8s ETA 4.2 min
5/10 41.3s avg 41.7s ETA 3.5 min
6/10 40.9s avg 41.6s ETA 2.8 min
7/10 41.6s avg 41.6s ETA 2.1 min
8/10 42.8s avg 41.8s ETA 1.4 min
9/10 42.0s avg 41.8s ETA 0.7 min
10/10 41.1s avg 41.7s ETA 0.0 min
checkpoint saved at 10/10
wrote /Users/raresolteanu/Desktop/Gliner-Work.Dauphine/communication_function_outputs_gemma_api_5d/gemma_api_annotations.csv
wrote /Users/raresolteanu/Desktop/Gliner-Work.Dauphine/communication_function_outputs_gemma_api_5d/gemma_api_errors.csv


,informativeness,expressiveness,conative,phatic,greenness,creativeness,metalingual,dominant_dimension,dominant_dimension_score,reason,confidence,model_dominant_dimension_raw,model_dominant_dimension_score_raw,row_id,model_name,parse_ok,raw_output,flagged
0,1.0,2.0,0.0,0.0,0.0,3.5,0.0,creativeness,3.5,Ad features a new ambassador and a creative fo...,0.90,creativeness,3.5,2100,mistral,True,"{\n ""informativeness"": 1.0,\n ""expressivene...",False
1,2.0,3.5,1.0,2.0,0.0,4.0,0.0,creativeness,4.0,Ad uses creative imagery and language to sell ...,0.95,creativeness,4.0,2101,mistral,True,"{\n ""informativeness"": 2.0,\n ""expressivene...",False
2,2.0,1.5,2.0,3.0,2.0,2.5,1.0,phatic,3.0,Ad emphasizes the constant readiness of the hy...,0.85,phatic,3.0,2102,mistral,True,"{\n ""informativeness"": 2.0,\n ""expressivene...",False
3,0.0,3.5,2.0,1.0,0.0,4.0,0.0,creativeness,4.0,Creative use of Wi-Fi as a metaphor for modern...,0.95,creativeness,4.0,2103,mistral,True,"{\n ""informativeness"": 0.0,\n ""expressivene...",False
4,2.0,4.0,3.0,1.0,0.0,4.0,0.0,mixed,4.0,Ad focuses on coolness and rivalry among adole...,0.95,creativeness,4.0,2104,mistral,True,"{\n ""informativeness"": 2.0,\n ""expressivene...",False


In [31]:
results_df["dominant_dimension"].value_counts(dropna=False)


dominant_dimension
creativeness      5
expressiveness    3
phatic            1
mixed             1
Name: count, dtype: int64

In [32]:
results_df["conative"].value_counts

<bound method IndexOpsMixin.value_counts of 0    0.0
1    1.0
2    2.0
3    2.0
4    3.0
5    1.0
6    0.0
7    2.0
8    2.0
9    1.0
Name: conative, dtype: float64>

In [33]:
results_df["creativeness"].value_counts

<bound method IndexOpsMixin.value_counts of 0    3.5
1    4.0
2    2.5
3    4.0
4    4.0
5    4.0
6    3.0
7    4.0
8    4.0
9    3.0
Name: creativeness, dtype: float64>

In [34]:
results_df[[
    "row_id",
    "informativeness",
    "expressiveness",
    "phatic",
    "greenness",
    "creativeness",
    "dominant_dimension",
    "reason",
    "confidence"
]].head(20)


,row_id,informativeness,expressiveness,phatic,greenness,creativeness,dominant_dimension,reason,confidence
0,2100,1.0,2.0,0.0,0.0,3.5,creativeness,Ad features a new ambassador and a creative fo...,0.90
1,2101,2.0,3.5,2.0,0.0,4.0,creativeness,Ad uses creative imagery and language to sell ...,0.95
2,2102,2.0,1.5,3.0,2.0,2.5,phatic,Ad emphasizes the constant readiness of the hy...,0.85
3,2103,0.0,3.5,1.0,0.0,4.0,creativeness,Creative use of Wi-Fi as a metaphor for modern...,0.95
4,2104,2.0,4.0,1.0,0.0,4.0,mixed,Ad focuses on coolness and rivalry among adole...,0.95
5,2105,2.0,3.5,2.0,0.0,4.0,creativeness,Ad uses creative imagery and storytelling to p...,0.95
6,2106,1.0,2.0,1.0,0.0,3.0,creativeness,Ad uses creative language and imagery to conve...,0.95
7,2107,3.0,4.5,1.0,2.0,4.0,expressiveness,Ad uses creative imagery and playful language ...,0.95
8,2108,3.0,4.5,1.0,3.0,4.0,expressiveness,Ad focuses on the new battery capacity of Rena...,0.95
9,2109,2.0,3.5,2.0,1.0,3.0,expressiveness,Ad focuses on the practicality and foreign sig...,0.90


In [35]:
results_df[
    results_df["dominant_dimension_score"] != results_df[DIMENSIONS].max(axis=1)
][[
    "row_id",
    "informativeness",
    "expressiveness",
    "phatic",
    "greenness",
    "creativeness",
    "dominant_dimension",
    "dominant_dimension_score"
]]


,row_id,informativeness,expressiveness,phatic,greenness,creativeness,dominant_dimension,dominant_dimension_score


In [36]:
results_df[[
    "row_id",
    "informativeness",
    "expressiveness",
    "phatic",
    "greenness",
    "creativeness",
    "dominant_dimension",
    "dominant_dimension_score",
    "confidence"
]].head(20)


,row_id,informativeness,expressiveness,phatic,greenness,creativeness,dominant_dimension,dominant_dimension_score,confidence
0,2100,1.0,2.0,0.0,0.0,3.5,creativeness,3.5,0.90
1,2101,2.0,3.5,2.0,0.0,4.0,creativeness,4.0,0.95
2,2102,2.0,1.5,3.0,2.0,2.5,phatic,3.0,0.85
3,2103,0.0,3.5,1.0,0.0,4.0,creativeness,4.0,0.95
4,2104,2.0,4.0,1.0,0.0,4.0,mixed,4.0,0.95
5,2105,2.0,3.5,2.0,0.0,4.0,creativeness,4.0,0.95
6,2106,1.0,2.0,1.0,0.0,3.0,creativeness,3.0,0.95
7,2107,3.0,4.5,1.0,2.0,4.0,expressiveness,4.5,0.95
8,2108,3.0,4.5,1.0,3.0,4.0,expressiveness,4.5,0.95
9,2109,2.0,3.5,2.0,1.0,3.0,expressiveness,3.5,0.90
